# 🧩 Modul 06: Unsupervised Learning I - Klasterisasi (*Clustering*) & Reduksi Dimensi (PCA)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antonprafanto/data-mining-zero-to-hero/blob/main/notebooks/06_Clustering_and_PCA.ipynb)
[![Course](https://img.shields.io/badge/Course-Data%20Mining%20Zero%20to%20Hero-blue.svg)](https://github.com/antonprafanto/data-mining-zero-to-hero)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

> **Pengampu**: Anton Prafanto  
> **Target Audiens**: Pemula tanpa latar belakang matematika/statistika tingkat lanjut.  
> **Prinsip Pembelajaran**: *Intuition before Formulas* — Pahami filosofi pengelompokan alami sebelum menyentuh persamaan jarak geometris.

---

### 🗺️ Peta Posisi Modul dalam CRISP-DM
```
[1. Business Understanding] ➔ [2. Data Understanding] ➔ [3. Data Preparation]
                                                               │
                                                               ▼
[6. Deployment] 💡 ◄── [5. Evaluation] ◄── 🎯 [4. MODELING: UNSUPERVISED]
```
Di Modul 04 dan 05 (*Supervised Learning*), kita memiliki kolom kunci jawaban / label target ($y$).  
Sekarang di **Modul 06**, kita memasuki belahan dunia baru: **Unsupervised Learning** — di mana mesin harus mencari pola tersembunyi, segmen pelanggan, dan merangkum dimensi data **tanpa bantuan kunci jawaban sama sekali**!


## 🕵️ 1. Filosofi Unsupervised Learning: Menemukan Pola Tanpa Kunci Jawaban

### 👶 Analogi Ramah Awam: Anak Kecil Merapikan Kotak Mainan
Bayangkan seorang anak balita yang diberikan sekotak mainan berantakan tanpa ada orang dewasa yang memberi instruksi:
* Anak tersebut tidak tahu nama resmi mainan itu (tidak ada label target $y$).
* Namun, secara alami anak itu mulai memilah:
  * Mobil-mobilan dikumpulkan di sudut kiri meja.
  * Balok kayu lego ditumpuk di tengah meja.
  * Boneka binatang diletakkan di sudut kanan meja.
* **Inilah esensi Unsupervised Clustering**: Mesin mengelompokkan data berdasarkan **tingkat kemiripan karakteristik alami (*similarity*)** antar titik data!

---

### ⚖️ Perbandingan 3 Paradigma Klasterisasi Populer:
| Algoritma | Filosofi Cara Kerja | Keunggulan Utama | Kelemahan / Batasan |
| :--- | :--- | :--- | :--- |
| **K-Means** | Menaruh $K$ titik pusat (*Centroid*) dan menarik tetangga terdekat. | Sangat cepat, efisien untuk data skala besar. | Bentuk klaster harus bulat/sferis, sensitif outlier. |
| **Hierarchical** | Menggabungkan pasangan data terdekat bertingkat (*Dendrogram*). | Menampilkan struktur hierarki silsilah keluarga data. | Boros memori untuk data berukuran raksasa ($O(n^2)$). |
| **DBSCAN** | Mengelompokkan titik berdasarkan kerapatan kerumunan (*Density*). | Mampu menemukan bentuk sembarang (bulan sabit) & kebal outlier (-1). | Butuh kalibrasi teliti untuk parameter `eps` & `min_samples`. |

---

### 📖 Kamus Istilah Awam (*Jargon Buster Modul 06*)
| Istilah Teknis | Bahasa Manusiawi / Analogi Sehari-hari |
| :--- | :--- |
| **Unsupervised Learning** | Belajar mandiri tanpa kunci jawaban (eksplorasi murni mencari pola tersembunyi). |
| **Centroid** | Titik pusat gravitasi dari suatu kelompok klaster. |
| **Inersia (WCSS)** | Jumlah total kuadrat jarak seluruh anggota klaster ke centroidnya (makin kecil = makin rapat). |
| **Elbow Method** | Grafik tekukan siku untuk mencari jumlah klaster paling efisien sebelum penurunannya melandai. |
| **Silhouette Score** | Skor kualitas klaster ($-1.0$ s/d $+1.0$): Mengukur seberapa rapat klaster sendiri dan seberapa jauh dari tetangga. |
| **Dendrogram** | Diagram pohon silsilah keluarga yang memperlihatkan alur penggabungan klaster bertahap. |
| **Linkage (Ward)** | Aturan penggabungan klaster yang meminimalkan kenaikan variansi internal. |
| **DBSCAN** | Algoritma kerumunan konser: mengelompokkan titik padat dan menandai orang yang menyendiri sebagai anomali. |
| **Epsilon (`eps`)** | Radius radar jangkauan lingkaran untuk mencari teman tetangga di DBSCAN. |
| **Noise (-1)** | Titik pencilan liar (*outlier*) yang diisolasi oleh DBSCAN karena tidak memiliki teman dekat. |
| **PCA** | Memotret patung 3D dari sudut terbaik agar bayangannya di dinding 2D tetap menampilkan esensi bentuknya. |
| **Explained Variance** | Persentase informasi asli yang berhasil diselamatkan setelah kolom dipangkas menjadi 2 sumbu komposit. |


## 🍷 2. Memuat Dataset Nyata: 13 Fitur Kimiawi Anggur (Wine Dataset)

Kita akan menggunakan **Wine Dataset** resmi dari Scikit-Learn. Dataset ini memuat **13 fitur kimiawi kontinu** dari sampel anggur (seperti kadar alkohol, asam malat, abu mineral, magnesium, flavonoid, intensitas warna, hingga kadar proline).

> ⚠️ **Hukum Wajib Klasterisasi**:  
> Seluruh algoritma klasterisasi berbasis jarak geometris (Euclidean) **WAJIB menggunakan Feature Scaling (StandardScaler)**!  
> Jika kadar proline bernilai 1.000 dan kadar asam malat bernilai 1.5, variabel proline akan mendominasi perhitungan jarak sebesar 99% dan membutakan variabel lainnya.


In [ ]:
# Import pustaka fundamental
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler

# Konfigurasi visualisasi
sns.set_theme(style='whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

# Memuat dataset anggur
wine = load_wine()
df_features = pd.DataFrame(wine.data, columns=wine.feature_names)

print(f"Dimensi Data: {df_features.shape[0]} sampel anggur × {df_features.shape[1]} fitur kimiawi")
print("\n5 Baris Sampel Fitur:")
df_features.head()


In [ ]:
# Standarisasi Seluruh Fitur (StandardScaler: Rata-rata = 0, Deviasi Standar = 1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_features)

print("✅ Standarisasi fitur berhasil dilakukan.")
print(f"Bentuk Matriks Berskala: {X_scaled.shape}")
print(f"Rata-rata fitur pertama setelah scale: {X_scaled[:, 0].mean():.4f} (Mendekati 0)")
print(f"Standar deviasi fitur pertama setelah scale: {X_scaled[:, 0].std():.4f} (Mendekati 1)")


## 🎯 3. K-Means Clustering: Membuka Gudang Logistik Baru

### 🚚 Analogi Dunia Nyata: Membuka $K$ Cabang Gudang
Bayangkan sebuah perusahaan logistik ingin membuka **$K=3$ cabang gudang baru**:
1. Perusahaan menaruh 3 titik acak (*Centroid*) di peta.
2. Setiap rumah pelanggan ditugaskan ke gudang terdekat.
3. Perusahaan menghitung ulang titik tengah dari seluruh pelanggan di masing-masing wilayah, lalu menggeser lokasi gudang ke titik tengah tersebut (*Update Centroids*).
4. Langkah 2 dan 3 diulang terus menerus hingga lokasi gudang tidak bergeser lagi (*Konvergensi*).

---

### 📉 Menentukan Jumlah $K$ Optimal: Dual Check (Elbow & Silhouette)
1. **Metode Siku (*Elbow Method*)**: Menghitung **Inersia** (WCSS - *Within-Cluster Sum of Squares*), yaitu total kuadrat jarak sampel ke titik pusatnya. Kita mencari titik "tekukan siku" di mana penurunan inersia mulai melambat drastis.
2. **Skor Silhouette (*Silhouette Score*)**: Mengukur seberapa rapat titik ke klaster sendiri ($a$) dibandingkan ke klaster tetangga ($b$). Nilai berkisar antara $-1.0$ sampai $+1.0$ (Mendekati $+1.0$ = klaster padat dan terpisah tegas).


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Uji variasi K dari 2 hingga 8
k_range = range(2, 9)
inertia_scores = []
sil_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia_scores.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(X_scaled, kmeans.labels_))

# Visualisasi Evaluasi Penentuan K Optimal (Dual Plot)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Plot 1: Elbow Method (Inersia WCSS)
axes[0].plot(k_range, inertia_scores, marker='o', color='royalblue', lw=2)
axes[0].axvline(x=3, color='crimson', linestyle='--', label='Tekukan Siku (K=3)')
axes[0].set_title("Metode Siku (Elbow Method)", fontweight='bold')
axes[0].set_xlabel("Jumlah Klaster (K)")
axes[0].set_ylabel("Inersia / WCSS")
axes[0].legend()
axes[0].grid(True, linestyle=':', alpha=0.6)

# Plot 2: Silhouette Score
axes[1].plot(k_range, sil_scores, marker='s', color='darkorange', lw=2)
axes[1].axvline(x=3, color='crimson', linestyle='--', label='Puncak Tertinggi (K=3)')
axes[1].set_title("Grafik Skor Silhouette", fontweight='bold')
axes[1].set_xlabel("Jumlah Klaster (K)")
axes[1].set_ylabel("Silhouette Score (-1 s/d +1)")
axes[1].legend()
axes[1].grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

# Latih Model Final dengan K=3
kmeans_final = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_kmeans = kmeans_final.fit_predict(X_scaled)

print(f"🏆 Rekomendasi Matematis: K = 3 adalah jumlah klaster paling optimal (Silhouette Score: {sil_scores[1]:.3f})!")


## 🌳 4. Hierarchical Clustering (Dendrogram)

### 👨‍👩‍👧‍👦 Analogi Pohon Silsilah Keluarga
Jika K-Means langsung membagi data ke dalam $K$ kelompok sekaligus, **Hierarchical Clustering (Agglomerative)** bekerja bertahap dari bawah ke atas (*Bottom-Up*):
1. Setiap sampel anggur awalnya dianggap sebagai **1 klaster independen**.
2. Dua sampel yang paling mirip digabungkan menjadi satu pasangan klaster.
3. Pasangan-pasangan tersebut terus digabungkan langkah demi langkah hingga akhirnya seluruh sampel bermuara pada satu batang pohon raksasa!

Bagan pohon ini disebut **Dendrogram**. Kita dapat memotong garis horizontal pada ketinggian tertentu untuk menentukan berapa banyak klaster yang terbentuk.


In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram

# Menghitung matriks jarak hierarki menggunakan metode 'ward' (meminimalkan variansi internal)
matriks_linkage = linkage(X_scaled, method='ward')

plt.figure(figsize=(14, 5.5))
dendro = dendrogram(
    matriks_linkage,
    truncate_mode='lastp',
    p=30, # Tampilkan 30 cabang teratas agar bersih
    leaf_rotation=90.,
    leaf_font_size=10.,
    show_contracted=True
)

# Garis potong ambang jarak (Threshold cut)
plt.axhline(y=10, color='crimson', linestyle='--', lw=2, label='Garis Potong Ambang Jarak (y=10 ➔ Menghasilkan 3 Klaster)')
plt.title("Dendrogram Hierarchical Clustering (Metode Ward)", fontsize=13, fontweight='bold', pad=12)
plt.xlabel("Indeks Klaster Sampel", fontsize=11)
plt.ylabel("Jarak Euklides Antar Klaster", fontsize=11)
plt.legend(loc='upper right')
plt.grid(True, linestyle=':', alpha=0.5)
plt.show()

print("Interpretasi Dendrogram:")
print("👉 Memotong dendrogram pada jarak y=10 membagi data menjadi 3 cabang warna utama (Hijau, Merah, Biru), konsisten dengan temuan K-Means!")


## 🌧️ 5. DBSCAN: Klaster Berbasis Kepadatan & Deteksi Outlier

### 🏟️ Analogi Kerumunan Penonton Konser
K-Means memiliki satu kelemahan fatal: **K-Means memaksa semua titik harus masuk ke salah satu klaster**, meskipun titik tersebut sebenarnya adalah data pencilan / outlier ekstrem!

Algoritma **DBSCAN (*Density-Based Spatial Clustering of Applications with Noise*)** bekerja dengan logika kerumunan:
* **Core Point**: Titik yang memiliki minimal `min_samples` tetangga di dalam radius lingkaran `eps`.
* **Border Point**: Titik di pinggiran kerumunan.
* **Noise / Outlier (Label = -1)**: Titik yang menyendiri jauh di sudut tribun tanpa tetangga dekat. Titik ini **tidak dipaksa masuk klaster**, melainkan langsung ditandai sebagai anomali!


In [ ]:
from sklearn.cluster import DBSCAN

# Latih DBSCAN dengan parameter radius eps dan batas minimal sampel
dbscan = DBSCAN(eps=2.3, min_samples=5)
cluster_dbscan = dbscan.fit_predict(X_scaled)

# Hitung sebaran klaster dan noise
unique_labels, counts = np.unique(cluster_dbscan, return_counts=True)
distribusi_dbscan = dict(zip(unique_labels, counts))

print("=== HASIL KLASTERISASI DBSCAN ===")
jumlah_klaster = len([k for k in unique_labels if k != -1])
jumlah_noise = distribusi_dbscan.get(-1, 0)

print(f"Jumlah Klaster Ditemukan : {jumlah_klaster} klaster padat")
print(f"Jumlah Titik Anomali/Noise : {jumlah_noise} sampel (diberi label -1)")
print(f"Distribusi Anggota Klaster : {distribusi_dbscan}")


## 📸 6. Reduksi Dimensi dengan PCA (*Principal Component Analysis*)

### 🗿 Analogi Memotret Patung 3D ke Foto 2D
Dataset Anggur kita memiliki **13 fitur kimiawi (13 dimensi)**. Mata manusia hanya bisa melihat grafik 2D atau 3D! Bagaimana cara kita memvisualisasikan sebaran klaster 13 dimensi di layar monitor?

Jawabannya adalah **PCA (Principal Component Analysis)**:
* Bayangkan sebuah patung 3 dimensi. Anda mencari sudut bidikan kamera paling strategis, sehingga bayangan patung di dinding 2D tetap memperlihatkan lekukan tubuh dan detail utama patung dengan jelas tanpa kehilangan esensi bentuknya.
* PCA memutar sumbu data dan membuat sumbu komposit baru (**Principal Component 1 & 2**) yang menangkap **persentase variansi data terbesar**!


In [ ]:
from sklearn.decomposition import PCA

# Reduksi 13 Dimensi menjadi 2 Dimensi Utama (PC1 & PC2)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

var_pc1 = pca.explained_variance_ratio_[0] * 100
var_pc2 = pca.explained_variance_ratio_[1] * 100
total_var = var_pc1 + var_pc2

print("=== REDUKSI DIMENSI PCA (13D ➔ 2D) ===")
print(f"Variansi PC1: {var_pc1:.2f}%")
print(f"Variansi PC2: {var_pc2:.2f}%")
print(f"Total Variansi Asli yang Berhasil Dipertahankan: {total_var:.2f}%!")

# Visualisasi Proyeksi 2D Klaster K-Means vs Klaster DBSCAN
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot K-Means pada Ruang PCA
scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_kmeans, cmap='Set1', s=45, alpha=0.8, edgecolor='black')
axes[0].set_title(f"Proyeksi 2D K-Means (K=3) - Terpisah Sangat Rapi!", fontweight='bold')
axes[0].set_xlabel(f"PC 1 ({var_pc1:.1f}% Variansi)")
axes[0].set_ylabel(f"PC 2 ({var_pc2:.1f}% Variansi)")
axes[0].grid(True, linestyle=':', alpha=0.5)

# Plot DBSCAN pada Ruang PCA (Outlier berlabel -1)
scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_dbscan, cmap='coolwarm', s=45, alpha=0.8, edgecolor='black')
axes[1].set_title("Proyeksi 2D DBSCAN (Titik Biru Tua = Anomali/Noise -1)", fontweight='bold')
axes[1].set_xlabel(f"PC 1 ({var_pc1:.1f}% Variansi)")
axes[1].set_ylabel(f"PC 2 ({var_pc2:.1f}% Variansi)")
axes[1].grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()


## 💼 7. Profiling Bisnis: Membaca Karakteristik Unik Setiap Klaster

Dalam industri data science, pekerjaan Anda **belum selesai** saat grafik klaster terbentuk. Pihak manajemen bisnis akan bertanya:  
*"Apa arti praktis dari Klaster 0, Klaster 1, dan Klaster 2 ini? Apa rekomendasi strategi bisnisnya?"*

Mari hitung rata-rata fitur kimiawi pada setiap klaster untuk menyusun profil bisnisnya!


In [ ]:
# Gabungkan label klaster kembali ke DataFrame asli
df_profil = df_features.copy()
df_profil['Klaster'] = cluster_kmeans

# Ambil fitur-fitur pembeda utama
fitur_kunci = ['alcohol', 'malic_acid', 'flavanoids', 'color_intensity', 'proline']
profil_klaster = df_profil.groupby('Klaster')[fitur_kunci].mean()

print("=== RATA-RATA FITUR KIMIAWI PER KLASTER ===")
print(profil_klaster.round(2))

print("\n📋 NAMA DAN REKOMENDASI SEGMEN BISNIS:")
print("🍇 Klaster 0: 'Anggur Ringan Ekonomis' -> Kadar alkohol dan proline terendah, warna lebih cerah.")
print("🍷 Klaster 1: 'Anggur Fermentasi Tajam' -> Asam malat tertinggi, flavonoid rendah (rasa tajam/sepat).")
print("🍾 Klaster 2: 'Anggur Sultan Premium'  -> Kadar alkohol (13.7%) dan proline tertinggi, warna sangat pekat.")


## 🧠 8. Kuis Evaluasi Pemahaman Mandiri

Uji pemahaman Anda terhadap konsep inti Unsupervised Learning, lalu jalankan sel kode di bawah untuk memeriksa kunci jawabannya:

---
* **Soal 1**: Mengapa pada K-Means dan DBSCAN kita WAJIB melakukan standarisasi fitur (*feature scaling*) sebelum melatih model?
* **Soal 2**: Apa kelemahan utama metode Elbow Method yang berhasil diatasi oleh analisis Silhouette Score?
* **Soal 3**: Mengapa algoritma DBSCAN sangat unggul dibandingkan K-Means pada data yang memiliki banyak pencilan liar (*noise*)?
* **Soal 4**: Apa arti nilai *Explained Variance Ratio* sebesar 55.4% pada proyeksi 2D PCA?
* **Soal 5**: Mengapa metode linkage 'ward' paling sering digunakan pada Hierarchical Clustering?


In [ ]:
# Kunci jawaban logis evaluasi mandiri Modul 6
kunci_jawaban_modul_6 = {
    "Soal 1": "Karena K-Means dan DBSCAN mengukur kemiripan berbasis jarak Euklides. Tanpa scaling, fitur berangka besar (misal kadar proline 1000) akan mendominasi perhitungan jarak sebesar 99% dan menenggelamkan fitur lainnya.",
    "Soal 2": "Elbow Method sering kali menghasilkan siku yang landai/kabur sehingga sulit menentukan titik belok secara objektif. Silhouette Score memberikan metrik numerik pasti (-1 hingga +1) terkait kepadatan dan pemisahan klaster.",
    "Soal 3": "K-Means memaksa seluruh sampel masuk ke dalam salah satu centroid sehingga outlier merusak titik pusat klaster. DBSCAN mendeteksi klaster berbasis kepadatan dan secara eksplisit melabeli outlier sebagai noise (-1).",
    "Soal 4": "Artinya dari 100% informasi/variabilitas yang tersebar di 13 dimensi asli, 55.4% di antaranya berhasil dirangkum dan dipertahankan hanya oleh 2 sumbu koordinat baru tersebut.",
    "Soal 5": "Metode linkage 'ward' meminimalkan kenaikan total variansi (Within-Cluster Sum of Squares) saat dua klaster digabungkan, sehingga menghasilkan kelompok yang padat dan ukuran klaster yang paling seimbang."
}

print("=== KUNCI JAWABAN EVALUASI MANDIRI MODUL 06 ===")
for soal, pembahasan in kunci_jawaban_modul_6.items():
    print(f"👉 {soal}:\n   {pembahasan}\n")


## ✍️ 9. Kotak Latihan Bebas (*Playground Challenge*)

### 🎯 Misi Eksperimen: Apa yang Terjadi Jika K-Means Dilatih Tanpa Standarisasi (*Unscaled*)?
Banyak pemula lupa melakukan `StandardScaler()` sebelum K-Means. Mari buktikan secara empiris mengapa ini adalah kesalahan fatal!

Tugas Anda:
1. Latih model K-Means pada data mentah `df_features` (tanpa standarisasi).
2. Hitung Silhouette Score-nya dan bandingkan dengan data yang sudah di-standarisasi (`X_scaled`).


In [ ]:
# Solusi Eksperimen: Membandingkan K-Means Tanpa Scaling vs Dengan Scaling
kmeans_mentah = KMeans(n_clusters=3, random_state=42, n_init=10).fit(df_features)
sil_mentah = silhouette_score(df_features, kmeans_mentah.labels_)
sil_berskala = silhouette_score(X_scaled, kmeans_final.labels_)

print("=== HASIL EKSPERIMEN STANDARISASI FITUR ===")
print(f"Silhouette Score Data Mentah (Tanpa Scaling) : {sil_mentah:.4f}")
print(f"Silhouette Score Data Berskala (StandardScaler): {sil_berskala:.4f}")

var_proline = df_features['proline'].var()
var_alcohol = df_features['alcohol'].var()

print(f"\n⚠️ Pada data mentah, fitur 'proline' memiliki variansi sebesar {var_proline:,.0f},")
print(f"sementara variansi 'alcohol' hanya sebesar {var_alcohol:.2f}!")
print("Akibatnya, klaster K-Means mentah sebetulnya HANYA mengelompokkan 1 fitur tersebut dan mengabaikan 12 fitur lainnya!")


## 🎯 10. Checklist Kelulusan Modul 06 (*Hero Checkpoint*)

Beri tanda centang pada hati Anda jika telah menguasai indikator penting berikut:
- [ ] Saya memahami bahwa Unsupervised Learning bekerja tanpa kolom kunci jawaban / label target ($y$).
- [ ] Saya paham mengapa standarisasi fitur (`StandardScaler`) adalah syarat wajib sebelum clustering.
- [ ] Saya bisa menentukan $K$ optimal pada K-Means menggunakan grafik **Elbow Method** dan **Silhouette Score**.
- [ ] Saya memahami cara membaca **Dendrogram** pada Hierarchical Clustering dan fungsi pemotongan ambang jarak.
- [ ] Saya mengerti keunggulan **DBSCAN** dalam mendeteksi klaster padat sekaligus melabeli anomali/noise (-1).
- [ ] Saya memahami konsep proyeksi 2D menggunakan **PCA** dan cara membaca *Explained Variance Ratio*.
- [ ] Saya mampu menerjemahkan angka rata-rata klaster menjadi label segmen bisnis yang nyata.

---

## 🏁 Gerbang Menuju Level Berikutnya: Unsupervised Learning II - Aturan Asosiasi!

Selamat! Anda kini telah resmi menguasai seni mengelompokkan data tanpa kunci jawaban serta mereduksi dimensi data menggunakan PCA!

Sekarang, bagaimana jika kita ingin mengetahui **perilaku belanja konsumen di kasir supermarket**?  
Produk apa yang paling sering dibeli secara bersamaan? Bagaimana toko ritel menata rak belanja agar omzet penjualan melonjak drastis?

Mari melangkah ke ranah **Unsupervised Learning II: Aturan Asosiasi (*Association Rule Mining*)**:

👉 **[Buka Modul 07: Unsupervised Learning II - Aturan Asosiasi](07_Association_Rule_Mining.ipynb)**  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antonprafanto/data-mining-zero-to-hero/blob/main/notebooks/07_Association_Rule_Mining.ipynb)

Sampai jumpa di Modul 07! Terus melaju menjadi Data Hero sejati! 🚀
